# Celda 1 - Configuración del entorno 

In [2]:

!pip install -q torch transformers datasets tokenizers evaluate accelerate


import torch
import transformers
import datasets
import random 


if torch.cuda.is_available():
    # Crea un objeto 'device' que apunte a la GPU principal
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f" Éxito: GPU detectada y configurada: {gpu_name}")
    print(f" Versión de CUDA disponible para PyTorch: {torch.version.cuda}")
else:

    print(" No se detectó una GPU ")
    device = torch.device("cpu")



# Limpia la caché de la GPU 
# util en recompilacion, comentar si es primera vez

if device.type == 'cuda': 
    torch.cuda.empty_cache()
    print(" Memoria de la GPU liberada y lista para trabajar.")


 No se detectó una GPU 



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Utilizamos CUDA, es lo mismo que ya se vio en clase
torch: Es la misma biblioteca que vimos  
transformers: Es la biblioteca de Hugging Face, tiene varias herramientas interesantes apra NLP de ahi vienen todas las siguientes herramientas 
datasets:para manejar conjuntos de datos
tokenizers:  para tokenizar texto 
evaluate: para evaluar modelos 
accelerate: para acelerar el entrenamiento de modelos mediante técnicas como mixed precision y distribución en múltiples GPUs

# Celda 2 Carga de datos y preprocesamiento

In [5]:
import os
from datasets import Dataset


ruta_es = "./Raw_Data/CCMatrix.es-la.es"
ruta_la = "./Raw_Data/CCMatrix.es-la.la"
ruta_scores = "./Raw_Data/CCMatrix.es-la.scores"




if all(os.path.exists(ruta) for ruta in [ruta_es, ruta_la, ruta_scores]):

    try:

        with open(ruta_es, 'r', encoding='utf-8') as f_es, \
             open(ruta_la, 'r', encoding='utf-8') as f_la, \
             open(ruta_scores, 'r', encoding='utf-8') as f_scores:
            
            # .strip() elimina los saltos de línea (\n) al final de cada oración
            textos_es = [linea.strip() for linea in f_es]
            textos_la = [linea.strip() for linea in f_la]
            # Convertimos los scores a números flotantes
            scores = [float(linea.strip()) for linea in f_scores]

        #  importante confirmar que el paralelismo es perfecto
        if not (len(textos_es) == len(textos_la) == len(scores)):
            raise ValueError("Los archivos no tienen la misma cantidad de líneas. El corpus está desalineado.")

        #  Construir el objeto Dataset de Hugging Face
        # Esto agrupa nuestras listas independientes en el formato optimizado que necesitamos
        dataset = Dataset.from_dict({
            "es": textos_es,
            "la": textos_la,
            "score": scores
        })

        print(f"\nDataset construido exitosamente desde los archivos paralelos :D ")
        print(f" Total de pares de oraciones disponibles en memoria: {len(dataset):,}")

        # muestreo 
        print("\n 5 Pares de Oraciones Aleatorias")
        indices_aleatorios = random.sample(range(len(dataset)), 5)

        for i, idx in enumerate(indices_aleatorios):
            fila = dataset[idx]
            print(f"\n Ejemplo {i+1} (Índice {idx}):")
            print(f"    Latín:   {fila['la']}")
            print(f"    Español: {fila['es']}")
            print(f"    Score:   {fila['score']}")

    except Exception as e:
        print(f"\n error {e}")

else:
    print("la ruta esta incorrecta o faltan archivos")


Dataset construido exitosamente desde los archivos paralelos :D 
 Total de pares de oraciones disponibles en memoria: 587,701

 5 Pares de Oraciones Aleatorias

 Ejemplo 1 (Índice 486690):
    Latín:   Nullam volutpat justo in mi tincidunt fermentum.
    Español: Una libélula aterrizó justo en mi tatuaje de libélula.
    Score:   1.0622262

 Ejemplo 2 (Índice 148797):
    Latín:   - Habesne fratres, sorores?
    Español: ¿Tenias hermanos o hermanas?
    Score:   1.0773771

 Ejemplo 3 (Índice 159311):
    Latín:   Scopri le nostre competenze.
    Español: cuenta nuestras aptitudes.
    Score:   1.0764427

 Ejemplo 4 (Índice 151256):
    Latín:   Curati e stai tranquillo!
    Español: ¡Cuídate y mantente sana!
    Score:   1.077153

 Ejemplo 5 (Índice 92182):
    Latín:   Tertio, quia de divinis non de facili debet homo aliter loqui quam sacra Scriptura loquatur.
    Español: En tercer lugar porque acerca de lo divino no debe el hombre con ligereza hablar de modo distinto a como lo hace

 Leer los archivos línea por línea simultáneamente
Usamos encoding='utf-8' para evitar problemas con tildes o caracteres especiales
la libreria Dataset es de Hugging Face se usa para organizar los datos de manera eficiente y compatible con modelos de NLP ( Natural Language Processing)
en este caso en especifico agrupa las listas de textos y scores en un formato tabular


# Celda 3: Filtrado de datos y limpieza

In [10]:
import re
import os


print("Iniciando el proceso de limpieza y filtrado del dataset...")

def limpiar_par_oraciones(fila): 
    import re 
    #importamos re adentro de la funcion porque windows no reconoce  el nucleo como global
    #solo para windows borrar si se usa en linux 
#  hiperparámetros de impieza 
    MIN_SCORE = 1.05
    MIN_PALABRAS = 3
    MAX_PALABRAS = 50
# Máxima desproporción permitida (ej. 2.0 significa que una oración 
# no puede tener más del doble de palabras que su contraparte)
    MAX_PROPORCION = 2.0

    # Regla 1: Filtro de puntuación de alineación
    if fila['score'] < MIN_SCORE:
        return False
    
    #fix: agregamos un filtro de caracteres especiales antes de la longitud para eliminar casos obvios de ruido
    
    patron_permitido = r'[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑüÜāēīōūĀĒĪŌŪ\s.,;:!?¿¡\'"()-]'
    
    texto_la = re.sub(patron_permitido, '', fila['la'])
    texto_es = re.sub(patron_permitido, '', fila['es'])
    
    if "http" in texto_la or "http" in texto_es or "<" in texto_la:
        return False
        
    texto_la = fila['la']
    texto_es = fila['es']
    
    #  filtro de caracteres 
    if "http" in texto_la or "http" in texto_es or "<" in texto_la:
        return False
    
      # filtro de longitud 
    palabras_la = texto_la.split()
    palabras_es = texto_es.split()
    
    len_la = len(palabras_la)
    len_es = len(palabras_es)
    
    if len_la < MIN_PALABRAS or len_la > MAX_PALABRAS:
        return False
    if len_es < MIN_PALABRAS or len_es > MAX_PALABRAS:
        return False
        

    proporcion = max(len_la, len_es) / min(len_la, len_es)
    if proporcion > MAX_PROPORCION:
        return False
        
    return True
        

# filtro con multiprocesamiento 

num_nucleos = os.cpu_count() or 2
dataset_limpio = dataset.filter(limpiar_par_oraciones, num_proc=num_nucleos)

print(f" Pares originales: {len(dataset):,}")
print(f" Pares limpios retenidos: {len(dataset_limpio):,}")
print(f" Pares descartados: {len(dataset) - len(dataset_limpio):,}")

Iniciando el proceso de limpieza y filtrado del dataset...


Filter (num_proc=12):   0%|          | 0/587701 [00:00<?, ? examples/s]

Filter (num_proc=12): 100%|██████████| 587701/587701 [00:14<00:00, 41765.90 examples/s] 

 Pares originales: 587,701
 Pares limpios retenidos: 521,461
 Pares descartados: 66,240



que significa es min score?
 Umbral de calidad del score (depende de cómo se generó CCMatrix, 
 valores superiores a 1.04 - 1.06 suelen indicar buena calidad en LASER

MAX_PROPORCION 
una regla de porque es importante usar esto es "GIGO" (basura entra basura sale) 
sin esto podria entrar una oracion en latin de 1 palabra y su contraparte 30, esto nos daria tremendas alucinadas
 2.0 significa que una oración 
 no puede tener más del doble de palabras que su contraparte)





In [12]:
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast
import os


print("Iniciando la configuración del Tokenizador BPE Compartido...")

tokenizador = ByteLevelBPETokenizer()

# generador para alimentar los datos

def iterador_de_textos(dataset, tamaño_lote=1000):
    for i in range(0, len(dataset), tamaño_lote):
        # Extraemos un lote y combinamos las listas de español y latín
        lote = dataset[i : i + tamaño_lote]
        yield lote["la"] + lote["es"]

# hiperparms del tokenizador
TAMAÑO_VOCABULARIO = 32000
FRECUENCIA_MINIMA = 2
TOKENS_ESPECIALES = [
    "<pad>", # Padding (relleno para igualar longitudes)
    "<bos>", # Beginning of sequence (inicio de oración)
    "<eos>", # End of sequence (fin de oración)
    "<unk>"  # Unknown (palabra desconocida)
]



#  tokenizador combinando ambos idiomas
tokenizador.train_from_iterator(
    iterador_de_textos(dataset_limpio),
    vocab_size=TAMAÑO_VOCABULARIO,
    min_frequency=FRECUENCIA_MINIMA,
    special_tokens=TOKENS_ESPECIALES
)

#  tokenizador en el entorno 
ruta_guardado = "./tokenizador_bpe_es_la"
if not os.path.exists(ruta_guardado):
    os.makedirs(ruta_guardado)

tokenizador.save(os.path.join(ruta_guardado, "tokenizer.json"))



Iniciando la configuración del Tokenizador BPE Compartido...


 Es un método de tokenización que divide las palabras en subpalabras, lo que permite manejar palabras desconocidas y reducir el tamaño del vocabulario, pero no es a nivel de caracter, en algunos articulos Se encontro que a caracter en NMT alucina bastante, con respuestas hasta 6 veces mas largas https://aclanthology.org/D18-1461/
 
 Tambien en este bloque envolvemos el tokenizador BPE en la clase de Hugging Face, esto nos permite usar funciones avanzadas requeridas por PyTorch (como tensores dinámicos) [tensores dinamicos: permiten manejar secuencias de longitud variable sin necesidad de padding excesivo ]

 attention_mask: una matriz binaria (de unos y ceros) que le indica a la  red neuronal qué tokens son palabras reales (1) y cuáles son simple relleno <pad> (0), para que la red no pierda tiempo de cómputo 
 https://machinelearningmastery.com/a-gentle-introduction-to-attention-masking-in-transformer-models/ 
 



# Fix Esta era una sola celda se dividio a 2 

In [15]:
import os
from transformers import PreTrainedTokenizerFast

print("Cargando el tokenizador entrenado y preparando las secuencias...")


def preparar_secuencias(lote):

    from transformers import PreTrainedTokenizerFast
    
    tokenizador_interno = PreTrainedTokenizerFast(
        tokenizer_file="./tokenizador_bpe_es_la/tokenizer.json",
        bos_token="<bos>",
        eos_token="<eos>",
        unk_token="<unk>",
        pad_token="<pad>"
    )
    
    LONGITUD_MAXIMA = 50 

    inputs = tokenizador_interno(
        lote["la"],
        max_length=LONGITUD_MAXIMA,
        padding="max_length", 
        truncation=True,      
    )
    

    labels = tokenizador_interno(
        lote["es"],
        max_length=LONGITUD_MAXIMA,
        padding="max_length",
        truncation=True,
    )
    
    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": labels["input_ids"]
    }

print("Aplicando la tokenización a todo el dataset limpio (num_proc=12)...")


num_nucleos = os.cpu_count() or 2
dataset_tokenizado = dataset_limpio.map(
    preparar_secuencias,
    batched=True,
    num_proc=num_nucleos,
    remove_columns=["la", "es", "score"] 
)
dataset_tokenizado.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

#  (90%) y Validación (10%) 
print("\nRealizando la división del corpus (90% Train / 10% Validation)...")
dataset_final = dataset_tokenizado.train_test_split(test_size=0.1, seed=42)

print("\n¡Dataset tokenizado y dividido con éxito!")
print("Estado final de los datos listos para la red:")
print(f"Lotes de Entrenamiento: {len(dataset_final['train']):,}") 
print(f"Lotes de Validación: {len(dataset_final['test']):,}") 

Cargando el tokenizador entrenado y preparando las secuencias...
Aplicando la tokenización a todo el dataset limpio (num_proc=12)...


Map (num_proc=12):   0%|          | 0/521461 [00:00<?, ? examples/s]

Map (num_proc=12): 100%|██████████| 521461/521461 [01:06<00:00, 7897.70 examples/s] 



Realizando la división del corpus (90% Train / 10% Validation)...

¡Dataset tokenizado y dividido con éxito!
Estado final de los datos listos para la red:
Lotes de Entrenamiento: 469,314
Lotes de Validación: 52,147


In [16]:
from transformers import BartConfig, BartForConditionalGeneration

print("Configurando la arquitectura del modelo Transformer...")

#  Hiperparámetros (Ajustados para 8GB de VRAM)
configuracion = BartConfig(
    vocab_size=32000,           # Debe coincidir exactamente con el tokenizador de la Celda 4
    d_model=512,                # Dimensión de los vectores 
    encoder_layers=6,           # Número de capas del codificador (
    decoder_layers=4,           # Número de capas del decodificador
    encoder_attention_heads=8,  # Cabezales de atención paralela
    decoder_attention_heads=8,
    encoder_ffn_dim=1024,       # Tamaño de la red feed-forward interna
    decoder_ffn_dim=1024,
    max_position_embeddings=50, # Límite de la secuencia (coincide con la Celda anterior)
    pad_token_id=tokenizador_hf.pad_token_id,
    bos_token_id=tokenizador_hf.bos_token_id,
    eos_token_id=tokenizador_hf.eos_token_id,
    forced_eos_token_id=tokenizador_hf.eos_token_id,
)


#  NO un modelo preentrenado. Los pesos nacen aleatorios.
modelo = BartForConditionalGeneration(configuracion)


# Movemos físicamente las matrices de la RAM del sistema a la VRAM de la RTX 4060
modelo = modelo.to(device)



total_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Total de parámetros  {total_params:,}")

Configurando la arquitectura del modelo Transformer...
Total de parámetros  41,673,728


Bart ("Bidirectional and Auto-Regressive Transformers") fue desarrollado por Facebook AI Research (FAIR) y es un modelo de lenguaje basado en la arquitectura Transformer. Es un modelo de secuencia a secuencia (seq2seq) [a cada secuencia de entrada le corresponde una secuencia de salida] que es justo lo que necesitamos en un traductor.
Bart por si solo no es un modelo preentrenado, es una arquitectura, lo que significa que no tiene pesos preentrenados, sino que se inicializa con pesos aleatorios. Esto es importante porque nos permite entrenar el modelo desde cero con nuestro propio conjunto de datos, sin depender de conocimientos previos que podrían no ser relevantes para la tarea específica de traducción entre latín y español. 
https://quantumailabs.net/guide-to-bart-bidirectional-autoregressive-transformer/
https://www.digitalocean.com/community/tutorials/bart-model-for-text-summarization-part1  
https://github.com/NVIDIA/DeepLearningExamples/tree/master/PyTorch/LanguageModeling/BART 

In [ ]:
import numpy as np
from transformers import Seq2SeqTrainingArguments

print("Configurando los hiperparámetros")

def calcular_metricas_convergencia(eval_pred):
    import numpy as np
    predicciones, etiquetas = eval_pred
    
    if isinstance(predicciones, tuple):
        logits = predicciones[0]
    else:
        logits = predicciones

    etiquetas = np.where(etiquetas != -100, etiquetas, 0)
    return {}


argumentos_entrenamiento = Seq2SeqTrainingArguments(
    output_dir="./modelo_latin_espanol", 
    

    learning_rate=1e-4,         
    warmup_steps=5000,               # Calentamiento adaptativo para proteger pesos iniciales aleatorios
    label_smoothing_factor=0.1,      # Suavizado contra ruido y paráfrasis de CCMatrix
    optim="adamw_torch",             

    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,   
    gradient_accumulation_steps=4,  
    fp16=True,                      
    

    eval_strategy="steps",           
    eval_steps=5000,                 # Frecuencia de evaluación intermedia
    save_strategy="steps",           
    save_steps=5000,                 
    load_best_model_at_end=True,     # Recupera automáticamente el punto exacto con menor pérdida
    metric_for_best_model="loss",    
    greater_is_better=False,         # Un valor menor de pérdida indica mejor traducción
    
    # Configuración de Inferencia en Validación
    predict_with_generate=True,      # Activa la generación real por búsqueda de haz
    report_to="none"                 
)


Configurando los hiperparámetros: Estrategia de Convergencia Segura...


estrategia de optimizacion basada en https://arxiv.org/abs/1706.03762
https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Seq2SeqTrainingArguments

5e-4 significa 0.0005, se bajo a 0.0001

In [23]:
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq, PreTrainedTokenizerFast

print("Preparando el motor de entrenamiento...")

# recargar el tokenizador en el entorno global por lo de windows
tokenizador_hf = PreTrainedTokenizerFast(
    tokenizer_file="./tokenizador_bpe_es_la/tokenizer.json",
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>"
)

#  Data Collator para tareas Seq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizador_hf, 
    model=modelo
)

# Trainer de Hugging Face
entrenador = Seq2SeqTrainer(
    model=modelo,                             # La arquitectura definida en la Celda 6
    args=argumentos_entrenamiento,            # Las reglas y optimizaciones de la Celda 7
    train_dataset=dataset_final["train"],     # El 90% de los datos para aprender
    eval_dataset=dataset_final["test"],       # El 10% de los datos para examinarse
    data_collator=data_collator,              # El empaquetador de matrices
    compute_metrics=calcular_metricas_convergencia, # Nuestra función de métricas
    processing_class=tokenizador_hf          # REEMPLAZO: Palabra clave moderna que corrige el TypeError
)


#  bucle de entrenamiento
resultados_entrenamiento = entrenador.train()


ruta_modelo_final = "./modelo_latin_espanol_final"
entrenador.save_model(ruta_modelo_final)


print(f"El modelo  guardado en: {ruta_modelo_final}")

Preparando el motor de entrenamiento...


a:\Python\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
a:\Python\Lib\site-packages\transformers\data\data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

expliquen la celda 8

In [ ]:
import torch
from transformers import BartForConditionalGeneration, PreTrainedTokenizerFast

print("Cargando el modelo final desde el disco y preparando la interfaz de traducción...")

# Cargar el tokenizador y el modelo final (Seguro para Windows y reinicios de kernel)
ruta_modelo = "./modelo_latin_espanol_final"
ruta_tokenizador = "./tokenizador_bpe_es_la/tokenizer.json"

tokenizador_inf = PreTrainedTokenizerFast(
    tokenizer_file=ruta_tokenizador,
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>"
)

# Cargamos la arquitectura con los pesos definitivos aprendidos por la GPU
modelo_inf = BartForConditionalGeneration.from_pretrained(ruta_modelo)

#  Transferir a la GPU y establecer el modo de evaluación
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_inf = modelo_inf.to(device)

# Cambia el comportamiento matemático de la red de "aprender" a "predecir"
modelo_inf.eval()

#  Definición de la función de inferencia
def traducir_latin_a_espanol(texto_latin):
    # Convertir el texto a tensores de PyTorch
    inputs = tokenizador_inf(
        texto_latin, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=40
    ).to(device)

    # Generar predicción bloqueando el cálculo de gradientes para ahorrar VRAM
    with torch.no_grad():
        outputs = modelo_inf.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=40,
            num_beams=5,          # Algoritmo de Búsqueda de Haz
            length_penalty=1.2,   # Penalización para favorecer la estructura del español
            early_stopping=True
        )

    # Decodificar los IDs numéricos devolviendo la cadena de texto limpia
    traduccion = tokenizador_inf.decode(outputs[0], skip_special_tokens=True)
    return traduccion

print("\nMotor de inferencia listo.")
print("Probando el modelo con una frase de ejemplo...")

# Prueba 
frase_prueba = input("\nIngrese una frase en latín para traducir al español: ")
traduccion_prueba = traducir_latin_a_espanol(frase_prueba)
print(f"\nLatín: {frase_prueba}")
print(f"Español: {traduccion_prueba}")

expliquen la celda 9

In [ ]:
import math
import evaluate
from tqdm.auto import tqdm

print("Iniciando la evaluación cuantitativa del modelo...")


# Le pedimos al motor que evalúe el 10% de datos que separamos de la celda 5
resultados_eval = entrenador.evaluate()
perdida_eval = resultados_eval.get("eval_loss", 0.0)

#  la perplejidad es la exponencial de la entropía cruzada
perplejidad = math.exp(perdida_eval) if perdida_eval > 0 else float('inf')

print("\n Resultados Internos de Convergencia (Salud de la Red):")
print(f" Pérdida de Validación (Cross-Entropy Loss): {perdida_eval:.4f}")
print(f" Perplejidad (Perplexity): {perplejidad:.4f}")


#  BLEU

metrica_bleu = evaluate.load("sacrebleu")

#  muestra aleatoria de 300 oraciones del test set
muestra = min(300, len(dataset_final["test"]))
muestra_test = dataset_final["test"].shuffle(seed=42).select(range(muestra))

predicciones_es = []
referencias_es = []

print(f"Traduciendo {muestra} oraciones de prueba para evaluar calidad...")

# Usamos nuestra función de inferencia de la Celda 9 de forma iterativa
for i in tqdm(range(muestra), desc="Generando Traducciones"):
    # Decodificamos los IDs numéricos de vuelta a texto para la evaluación
    texto_lat = tokenizador_inf.decode(muestra_test[i]["input_ids"], skip_special_tokens=True)
    referencia_humana = tokenizador_inf.decode(muestra_test[i]["labels"], skip_special_tokens=True)
    
    # Generamos la traducción con la red neuronal
    prediccion_maquina = traducir_latin_a_espanol(texto_lat)
    
    predicciones_es.append(prediccion_maquina)
    # SacreBLEU requiere que las referencias estén en formato de lista de listas
    referencias_es.append([referencia_humana]) 

resultados_bleu = metrica_bleu.compute(predictions=predicciones_es, references=referencias_es)

print("\n Resultados Externos de Calidad de Traducción:")
print(f" Puntuación BLEU: {resultados_bleu['score']:.2f}")


explicacion de la celda anterior aqui

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


# Esta matriz contiene la representación vectorial de las 32,000 subpalabras
matriz_embeddings = modelo_inf.get_input_embeddings().weight.detach().cpu().numpy()

#  Definir un conjunto de raíces y cognados de interés para analizar
# Usamos el prefijo 'Ġ' (espacio) o sin él dependiendo de cómo BPE fragmentó las palabras.
palabras_interes = [
    "Deus", "Dios", 
    "am", "amor", "amare",
    "nat", "naturaleza", "natura",
    "dict", "diccionario", "dictum",
    "form", "forma",
    "pater", "padre",
    "mater", "madre",
    "fili", "hijo",
    "sanct", "santo"
]

ids_interes = []
etiquetas_encontradas = []

print("Buscando los identificadores de los tokens en el vocabulario BPE...")

# 3. Buscar los IDs numéricos de estas palabras en nuestro tokenizador
for palabra in palabras_interes:
    # Codificamos la palabra sin añadir <bos> ni <eos>
    tokens = tokenizador_inf.encode(palabra, add_special_tokens=False)
    if len(tokens) > 0:
        # Tomamos el ID de la primera subpalabra o raíz principal
        ids_interes.append(tokens[0])
        etiquetas_encontradas.append(palabra)

# Extraer exclusivamente los vectores de 256 dimensiones de los tokens encontrados
vectores_interes = matriz_embeddings[ids_interes]

# 4. Reducir la dimensionalidad de 256D a 2D utilizando PCA
print("Reduciendo dimensiones con PCA para graficar en 2D...")
pca = PCA(n_components=2)
vectores_2d = pca.fit_transform(vectores_interes)

# 5. Generar la gráfica interactiva
plt.figure(figsize=(12, 8))
plt.scatter(vectores_2d[:, 0], vectores_2d[:, 1], color='royalblue', edgecolors='black', s=80, alpha=0.7)

# Añadir las etiquetas a cada punto
for i, etiqueta in enumerate(etiquetas_encontradas):
    plt.annotate(
        etiqueta, 
        (vectores_2d[i, 0], vectores_2d[i, 1]), 
        xytext=(8, 5), 
        textcoords='offset points', 
        fontsize=11,
        fontweight='bold'
    )

plt.title("Espacio Latente Compartido: Cognados Latín-Español", fontsize=16, pad=15)
plt.xlabel("Componente Principal 1", fontsize=12)
plt.ylabel("Componente Principal 2", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)


plt.show()

In [ ]:
celda 11 explicacion

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch

print("Generando visualizaciones avanzadas de rendimiento...\n")


try:

    historial = entrenador.state.log_history

    pasos_train = []
    loss_train = []
    pasos_eval = []
    loss_eval = []

    for log in historial:
        if 'loss' in log and 'step' in log:
            pasos_train.append(log['step'])
            loss_train.append(log['loss'])
        elif 'eval_loss' in log and 'step' in log:
            pasos_eval.append(log['step'])
            loss_eval.append(log['eval_loss'])

    plt.figure(figsize=(10, 6))
    if loss_train:
        plt.plot(pasos_train, loss_train, label='Pérdida de Entrenamiento (Train Loss)', color='royalblue', alpha=0.8)
    if loss_eval:
        plt.plot(pasos_eval, loss_eval, label='Pérdida de Validación (Eval Loss)', color='crimson', marker='o', linewidth=2)

    plt.title('Curvas de Convergencia NMT: Latín a Español', fontsize=16, pad=15)
    plt.xlabel('Pasos de Entrenamiento', fontsize=12)
    plt.ylabel('Entropía Cruzada (Loss)', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

except NameError:
    print(" ya no está en memoria. ")


print("\n  Generando Mapa de Atención Cruzada (Alineación de Modelos)...")

frase_atencion = "Alea iacta est."

inputs = tokenizador_inf(frase_atencion, return_tensors="pt").to(device)


with torch.no_grad():
    outputs = modelo_inf.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=20,
        output_attentions=True,           # Extrae los pesos de atención
        return_dict_in_generate=True      # Devuelve un objeto estructurado
    )

# Decodificamos los IDs numéricos de vuelta a texto para los ejes X y Y
tokens_latin = tokenizador_inf.convert_ids_to_tokens(inputs["input_ids"][0])
tokens_espanol = tokenizador_inf.convert_ids_to_tokens(outputs.sequences[0])

# Limpiamos el marcador de BPE ('Ġ' o espacios) para que la gráfica se lea bien
tokens_latin = [t.replace('Ġ', '').replace(' ', '') for t in tokens_latin]
tokens_espanol = [t.replace('Ġ', '').replace(' ', '') for t in tokens_espanol]

try:
    # outputs.cross_attentions contiene la atención del decodificador hacia el codificador
    atenciones = outputs.cross_attentions
    
    # Construimos una matriz vacía: [Palabras Generadas (Español) vs Palabras Origen (Latín)]
    # Omitimos el primer token generado porque suele ser el <bos> (inicio de oración)
    matriz_atencion = np.zeros((len(tokens_espanol) - 1, len(tokens_latin)))

    # Extraemos y promediamos matemáticamente los cabezales (Multi-Head)
    for i, tupla_atencion_paso in enumerate(atenciones):
        # Tomamos la última capa del decodificador [-1] del batch actual [0]
        capa_final = tupla_atencion_paso[-1][0] 
        # Promediamos el enfoque de los 8 cabezales de atención
        promedio_cabezales = capa_final.mean(dim=0).squeeze().cpu().numpy()
        matriz_atencion[i] = promedio_cabezales

    # Creación del Mapa de Calor (Heatmap)
    plt.figure(figsize=(10, 8))
    sns.heatmap(matriz_atencion, 
                xticklabels=tokens_latin, 
                yticklabels=tokens_espanol[1:], 
                cmap="mako", 
                annot=True,       # Muestra el valor probabilístico exacto
                fmt=".2f", 
                cbar_kws={'label': 'Peso de Atención'})
    
    plt.title('Mecanismo de Autoatención: Alineación Latín-Español', fontsize=16, pad=15)
    plt.xlabel('Texto de Origen (Latín)', fontsize=12, fontweight='bold')
    plt.ylabel('Texto Generado (Español)', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.show()

except Exception as e:
    print(f"Nota: Tu arquitectura específica no devolvió 'cross_attentions'. Detalle técnico: {e}")

celda 12 analisis visual 